# PMRAM MRI MobileNetV2 Ablation Study

This notebook is prepared from your original hybrid file. Your original CNN backbone is **MobileNetV2**.

Loss modes included:

1. `ce` = standard categorical cross-entropy.
2. `custom` = cross-entropy + capsule/margin-style loss.

Change `ZIP_PATH` if your dataset zip name is different in Colab.

## This file runs

`mobilenetv2_only, mobilenetv2_transformer, mobilenetv2_capsule, mobilenetv2_transformer_capsule`


In [ ]:
# ============================================================
# 1. Imports, seed, and GPU check
# ============================================================
import os
import glob
import random
import zipfile
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_v2_preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.metrics import AUC
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, precision_recall_fscore_support

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


In [ ]:
# ============================================================
# 2. Dataset extraction and path setup
# ============================================================
# Your original notebook used: /content/archive (53).zip
# If your zip has a different name, either update ZIP_PATH or upload only one zip to /content.
ZIP_PATH = "/content/archive (53).zip"
EXTRACT_PATH = "/content/extracted_data"

if not os.path.exists(ZIP_PATH):
    zip_candidates = glob.glob("/content/*.zip")
    if len(zip_candidates) == 1:
        ZIP_PATH = zip_candidates[0]
        print("Auto-detected zip:", ZIP_PATH)
    else:
        print("ZIP not found at default path.")
        print("Available zip files:", zip_candidates)
        print("Upload your dataset zip to Colab or update ZIP_PATH manually.")

if os.path.exists(ZIP_PATH):
    os.makedirs(EXTRACT_PATH, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print("Extracted to:", EXTRACT_PATH)
    print("Top-level files/folders:", os.listdir(EXTRACT_PATH))

# Original paths from your notebook
TRAIN_PATH = "/content/extracted_data/PMRAM Bangladeshi Brain Cancer - MRI Dataset/PMRAM Bangladeshi Brain Cancer - MRI Dataset/Augmented Data/Augmented"
TEST_PATH = "/content/extracted_data/PMRAM Bangladeshi Brain Cancer - MRI Dataset/PMRAM Bangladeshi Brain Cancer - MRI Dataset/Raw Data/Raw"

# Fallback auto-detection if the folder level changes after extraction
def find_dataset_folder(root, required_text):
    matches = []
    for dirpath, dirnames, filenames in os.walk(root):
        normalized = dirpath.replace('\\', '/').lower()
        if required_text.lower() in normalized:
            # A valid ImageFolder-like directory should contain class subfolders
            subdirs = [d for d in os.listdir(dirpath) if os.path.isdir(os.path.join(dirpath, d))]
            if len(subdirs) >= 2:
                matches.append(dirpath)
    return sorted(matches, key=len)

if not os.path.isdir(TRAIN_PATH):
    candidates = find_dataset_folder(EXTRACT_PATH, "Augmented Data/Augmented")
    if candidates:
        TRAIN_PATH = candidates[0]

if not os.path.isdir(TEST_PATH):
    candidates = find_dataset_folder(EXTRACT_PATH, "Raw Data/Raw")
    if candidates:
        TEST_PATH = candidates[0]

print("TRAIN_PATH:", TRAIN_PATH, "exists=", os.path.isdir(TRAIN_PATH))
print("TEST_PATH:", TEST_PATH, "exists=", os.path.isdir(TEST_PATH))


In [ ]:
# ============================================================
# 3. Global configuration
# ============================================================
IMAGE_SIZE = (224, 224)
INPUT_SHAPE = (224, 224, 3)
BATCH_SIZE = 16

# Transformer settings
EMBED_DIM = 96
NUM_HEADS = 4
TRANSFORMER_DEPTH = 1
MLP_DIM = 192

# Capsule settings
PRIMARY_CAPS = 12
PRIMARY_CAPS_DIM = 8
CLASS_CAPS_DIM = 12
ROUTING_ITERS = 2

# Training and loss settings
DROPOUT = 0.30
LABEL_SMOOTHING = 0.08
REG = tf.keras.regularizers.l2(1e-4)
ALPHA_CE = 0.9
BETA_MARGIN = 0.1

PHASE1_EPOCHS = 15
PHASE2_EPOCHS = 15
PHASE1_LR = 1e-3
PHASE2_LR = 5e-5
WEIGHT_DECAY = 1e-4

os.makedirs('/content/working', exist_ok=True)


In [ ]:
# ============================================================
# 4. Data generators
# ============================================================
def build_generators(train_path, test_path, image_size=(224, 224), batch_size=16, validation_split=0.2):
    train_datagen = ImageDataGenerator(
        preprocessing_function=mobilenet_v2_preprocess_input,
        rotation_range=10,
        width_shift_range=0.05,
        height_shift_range=0.05,
        zoom_range=0.06,
        brightness_range=(0.95, 1.05),
        fill_mode='nearest',
        validation_split=validation_split
    )

    val_datagen = ImageDataGenerator(
        preprocessing_function=mobilenet_v2_preprocess_input,
        validation_split=validation_split
    )

    test_datagen = ImageDataGenerator(
        preprocessing_function=mobilenet_v2_preprocess_input
    )

    train_gen = train_datagen.flow_from_directory(
        train_path,
        target_size=image_size,
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=True,
        subset='training',
        seed=SEED
    )

    val_gen = val_datagen.flow_from_directory(
        train_path,
        target_size=image_size,
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=False,
        subset='validation',
        seed=SEED
    )

    test_gen = test_datagen.flow_from_directory(
        test_path,
        target_size=image_size,
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=False
    )
    return train_gen, val_gen, test_gen

train_gen, val_gen, test_gen = build_generators(
    TRAIN_PATH,
    TEST_PATH,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=0.2
)

CLASS_NAMES = list(train_gen.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)
print("Class indices:", train_gen.class_indices)
print("NUM_CLASSES:", NUM_CLASSES)


In [ ]:
# ============================================================
# 5. Shared custom layers: patch tokenization, transformer, capsule
# ============================================================
def squash(vectors, axis=-1, epsilon=1e-7):
    s_squared_norm = tf.reduce_sum(tf.square(vectors), axis=axis, keepdims=True)
    scale = s_squared_norm / (1.0 + s_squared_norm)
    return scale * vectors / tf.sqrt(s_squared_norm + epsilon)


class PatchTokenization(layers.Layer):
    def __init__(self, patch_size=2, embed_dim=EMBED_DIM, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.proj = layers.Dense(embed_dim, kernel_regularizer=REG)

    def build(self, input_shape):
        _, h, w, c = input_shape
        self.num_patches_h = h // self.patch_size
        self.num_patches_w = w // self.patch_size
        self.num_tokens = self.num_patches_h * self.num_patches_w
        self.pos_embed = self.add_weight(
            name='pos_embed',
            shape=(1, self.num_tokens, self.embed_dim),
            initializer='random_normal',
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        patches = tf.image.extract_patches(
            images=x,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID'
        )
        batch_size = tf.shape(patches)[0]
        patch_dim = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, self.num_tokens, patch_dim])
        tokens = self.proj(patches)
        return tokens + self.pos_embed


class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=DROPOUT, **kwargs):
        super().__init__(**kwargs)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout
        )
        self.drop1 = layers.Dropout(dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = tf.keras.Sequential([
            layers.Dense(mlp_dim, activation='gelu', kernel_regularizer=REG),
            layers.Dropout(dropout),
            layers.Dense(embed_dim, kernel_regularizer=REG),
            layers.Dropout(dropout)
        ])

    def call(self, x, training=False, return_attention=False):
        x_norm = self.norm1(x)
        attn_out, attn_scores = self.attn(
            x_norm, x_norm,
            return_attention_scores=True,
            training=training
        )
        x = x + self.drop1(attn_out, training=training)
        x = x + self.mlp(self.norm2(x), training=training)
        if return_attention:
            return x, attn_scores
        return x


class PrimaryCapsule(layers.Layer):
    def __init__(self, num_capsules=PRIMARY_CAPS, capsule_dim=PRIMARY_CAPS_DIM, **kwargs):
        super().__init__(**kwargs)
        self.num_capsules = num_capsules
        self.capsule_dim = capsule_dim
        self.proj = layers.Dense(num_capsules * capsule_dim, kernel_regularizer=REG)

    def call(self, tokens):
        x = self.proj(tokens)
        batch_size = tf.shape(x)[0]
        num_tokens = tf.shape(x)[1]
        x = tf.reshape(x, [batch_size, num_tokens, self.num_capsules, self.capsule_dim])
        return squash(x)


class RelevanceAwareClassCapsule(layers.Layer):
    def __init__(self, num_classes, class_caps_dim=CLASS_CAPS_DIM, routing_iters=ROUTING_ITERS, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.class_caps_dim = class_caps_dim
        self.routing_iters = routing_iters

    def build(self, input_shape):
        _, num_tokens, num_primary, primary_dim = input_shape[0]
        self.W = self.add_weight(
            shape=(1, num_tokens, num_primary, self.num_classes, self.class_caps_dim, primary_dim),
            initializer='glorot_uniform',
            trainable=True,
            name='capsule_transform'
        )
        super().build(input_shape)

    def call(self, inputs):
        primary_caps, relevance_scores = inputs
        primary_caps_exp = tf.expand_dims(tf.expand_dims(primary_caps, axis=3), axis=-1)
        W_tiled = tf.tile(self.W, [tf.shape(primary_caps)[0], 1, 1, 1, 1, 1])
        u_hat = tf.matmul(W_tiled, primary_caps_exp)
        u_hat = tf.squeeze(u_hat, axis=-1)

        b = tf.zeros(
            shape=(tf.shape(primary_caps)[0], tf.shape(primary_caps)[1], tf.shape(primary_caps)[2], self.num_classes),
            dtype=tf.float32
        )
        relevance = tf.expand_dims(tf.expand_dims(relevance_scores, axis=-1), axis=-1)

        for i in range(self.routing_iters):
            c = tf.nn.softmax(b, axis=-1)
            c = c * relevance
            c = c / (tf.reduce_sum(c, axis=-1, keepdims=True) + 1e-8)
            s = tf.reduce_sum(tf.expand_dims(c, axis=-1) * u_hat, axis=[1, 2])
            v = squash(s)
            if i < self.routing_iters - 1:
                agreement = tf.reduce_sum(
                    u_hat * tf.expand_dims(tf.expand_dims(v, axis=1), axis=1),
                    axis=-1
                )
                b = b + agreement
        return v, c


In [ ]:
# ============================================================
# 6. Model definitions
# ============================================================
class MobileNetV2Only(Model):
    """MobileNetV2 CNN only. No Transformer. No Capsule."""
    def __init__(self, num_classes, dropout=DROPOUT, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.backbone = MobileNetV2(weights='imagenet', include_top=False, input_shape=INPUT_SHAPE)
        self.backbone.trainable = False
        self.pool = layers.GlobalAveragePooling2D()
        self.dropout = layers.Dropout(dropout)
        self.classifier = layers.Dense(num_classes, activation='softmax', kernel_regularizer=REG)

    def call(self, inputs, training=False, return_extras=False):
        feature_map = self.backbone(inputs, training=training)
        x = self.pool(feature_map)
        x = self.dropout(x, training=training)
        probs = self.classifier(x)
        if return_extras:
            return {'logits': probs, 'feature_map': feature_map}
        return probs


class MobileNetV2Transformer(Model):
    """MobileNetV2 + custom Transformer encoder. No Capsule."""
    def __init__(self, num_classes, embed_dim=EMBED_DIM, num_heads=NUM_HEADS,
                 transformer_depth=TRANSFORMER_DEPTH, mlp_dim=MLP_DIM, dropout=DROPOUT, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.backbone = MobileNetV2(weights='imagenet', include_top=False, input_shape=INPUT_SHAPE)
        self.backbone.trainable = False
        self.proj_conv = layers.Conv2D(embed_dim, 1, padding='same', use_bias=False, kernel_regularizer=REG)
        self.proj_bn = layers.BatchNormalization()
        self.proj_act = layers.Activation('swish')
        self.proj_dropout = layers.Dropout(0.25)
        self.tokenizer = PatchTokenization(patch_size=2, embed_dim=embed_dim)
        self.transformer_blocks = [
            TransformerBlock(embed_dim, num_heads, mlp_dim, dropout=dropout, name=f'transformer_block_{i}')
            for i in range(transformer_depth)
        ]
        self.final_norm = layers.LayerNormalization(epsilon=1e-6)
        self.pool = layers.GlobalAveragePooling1D()
        self.dropout = layers.Dropout(dropout)
        self.classifier = layers.Dense(num_classes, activation='softmax', kernel_regularizer=REG)
        self.last_attention_scores = None

    def call(self, inputs, training=False, return_extras=False):
        feature_map = self.backbone(inputs, training=training)
        feature_map = self.proj_act(self.proj_bn(self.proj_conv(feature_map), training=training))
        feature_map = self.proj_dropout(feature_map, training=training)
        tokens = self.tokenizer(feature_map)

        attention_scores_list = []
        for i, blk in enumerate(self.transformer_blocks):
            if i == len(self.transformer_blocks) - 1:
                tokens, attn = blk(tokens, training=training, return_attention=True)
                attention_scores_list.append(attn)
            else:
                tokens = blk(tokens, training=training)

        tokens = self.final_norm(tokens)
        pooled = self.pool(tokens)
        pooled = self.dropout(pooled, training=training)
        probs = self.classifier(pooled)
        self.last_attention_scores = attention_scores_list[-1] if attention_scores_list else None

        if return_extras:
            return {
                'logits': probs,
                'feature_map': feature_map,
                'tokens': tokens,
                'attention_scores': self.last_attention_scores
            }
        return probs


class MobileNetV2Capsule(Model):
    """MobileNetV2 CNN + Capsule. No Transformer."""
    def __init__(self, num_classes, embed_dim=EMBED_DIM, primary_caps=PRIMARY_CAPS,
                 primary_caps_dim=PRIMARY_CAPS_DIM, class_caps_dim=CLASS_CAPS_DIM,
                 routing_iters=ROUTING_ITERS, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.backbone = MobileNetV2(weights='imagenet', include_top=False, input_shape=INPUT_SHAPE)
        self.backbone.trainable = False
        self.proj_conv = layers.Conv2D(embed_dim, 1, padding='same', use_bias=False, kernel_regularizer=REG)
        self.proj_bn = layers.BatchNormalization()
        self.proj_act = layers.Activation('swish')
        self.proj_dropout = layers.Dropout(0.25)
        self.tokenizer = PatchTokenization(patch_size=2, embed_dim=embed_dim)
        self.token_relevance_head = layers.Dense(1, activation='sigmoid', kernel_regularizer=REG)
        self.primary_caps = PrimaryCapsule(primary_caps, primary_caps_dim)
        self.class_caps = RelevanceAwareClassCapsule(num_classes, class_caps_dim, routing_iters)
        self.last_routing_coeffs = None

    def call(self, inputs, training=False, return_extras=False):
        feature_map = self.backbone(inputs, training=training)
        feature_map = self.proj_act(self.proj_bn(self.proj_conv(feature_map), training=training))
        feature_map = self.proj_dropout(feature_map, training=training)
        tokens = self.tokenizer(feature_map)
        token_relevance = tf.squeeze(self.token_relevance_head(tokens), axis=-1)
        token_relevance = token_relevance / (tf.reduce_sum(token_relevance, axis=-1, keepdims=True) + 1e-8)
        primary_caps = self.primary_caps(tokens)
        class_caps, routing_coeffs = self.class_caps([primary_caps, token_relevance])
        caps_lengths = tf.norm(class_caps, axis=-1)
        probs = tf.nn.softmax(caps_lengths, axis=-1)
        self.last_routing_coeffs = routing_coeffs
        if return_extras:
            return {
                'logits': probs,
                'caps_lengths': caps_lengths,
                'class_capsules': class_caps,
                'feature_map': feature_map,
                'tokens': tokens,
                'relevance': token_relevance,
                'routing_coeffs': routing_coeffs
            }
        return probs


class MobileNetV2TransformerCapsule(Model):
    """MobileNetV2 + custom Transformer encoder + Capsule. This matches your full hybrid idea."""
    def __init__(self, num_classes, embed_dim=EMBED_DIM, num_heads=NUM_HEADS,
                 transformer_depth=TRANSFORMER_DEPTH, mlp_dim=MLP_DIM,
                 primary_caps=PRIMARY_CAPS, primary_caps_dim=PRIMARY_CAPS_DIM,
                 class_caps_dim=CLASS_CAPS_DIM, routing_iters=ROUTING_ITERS, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.backbone = MobileNetV2(weights='imagenet', include_top=False, input_shape=INPUT_SHAPE)
        self.backbone.trainable = False
        self.proj_conv = layers.Conv2D(embed_dim, 1, padding='same', use_bias=False, kernel_regularizer=REG)
        self.proj_bn = layers.BatchNormalization()
        self.proj_act = layers.Activation('swish')
        self.proj_dropout = layers.Dropout(0.25)
        self.tokenizer = PatchTokenization(patch_size=2, embed_dim=embed_dim)
        self.transformer_blocks = [
            TransformerBlock(embed_dim, num_heads, mlp_dim, dropout=DROPOUT, name=f'transformer_block_{i}')
            for i in range(transformer_depth)
        ]
        self.final_norm = layers.LayerNormalization(epsilon=1e-6)
        self.token_relevance_head = layers.Dense(1, activation='sigmoid', kernel_regularizer=REG)
        self.primary_caps = PrimaryCapsule(primary_caps, primary_caps_dim)
        self.class_caps = RelevanceAwareClassCapsule(num_classes, class_caps_dim, routing_iters)
        self.last_attention_scores = None
        self.last_routing_coeffs = None

    def call(self, inputs, training=False, return_extras=False):
        feature_map = self.backbone(inputs, training=training)
        feature_map = self.proj_act(self.proj_bn(self.proj_conv(feature_map), training=training))
        feature_map = self.proj_dropout(feature_map, training=training)
        tokens = self.tokenizer(feature_map)

        attention_scores_list = []
        for i, blk in enumerate(self.transformer_blocks):
            if i == len(self.transformer_blocks) - 1:
                tokens, attn = blk(tokens, training=training, return_attention=True)
                attention_scores_list.append(attn)
            else:
                tokens = blk(tokens, training=training)

        tokens = self.final_norm(tokens)
        token_relevance = tf.squeeze(self.token_relevance_head(tokens), axis=-1)

        if attention_scores_list:
            last_attn = attention_scores_list[-1]
            mean_attn = tf.reduce_mean(last_attn, axis=1)
            attn_importance = tf.reduce_mean(mean_attn, axis=1)
            fused_relevance = 0.5 * attn_importance + 0.5 * token_relevance
        else:
            fused_relevance = token_relevance

        fused_relevance = fused_relevance / (tf.reduce_sum(fused_relevance, axis=-1, keepdims=True) + 1e-8)
        primary_caps = self.primary_caps(tokens)
        class_caps, routing_coeffs = self.class_caps([primary_caps, fused_relevance])
        caps_lengths = tf.norm(class_caps, axis=-1)
        probs = tf.nn.softmax(caps_lengths, axis=-1)

        self.last_attention_scores = attention_scores_list[-1] if attention_scores_list else None
        self.last_routing_coeffs = routing_coeffs

        if return_extras:
            return {
                'logits': probs,
                'caps_lengths': caps_lengths,
                'class_capsules': class_caps,
                'feature_map': feature_map,
                'tokens': tokens,
                'relevance': fused_relevance,
                'attention_scores': self.last_attention_scores,
                'routing_coeffs': routing_coeffs
            }
        return probs


In [ ]:
# ============================================================
# 7. Custom loss trainer, training function, and evaluation
# ============================================================
class CustomLossTrainer(Model):
    """Trainer for CE + capsule/margin-style loss.

    For capsule models, this is the same idea as your original CE + Capsule Loss.
    For non-capsule models, the margin component is applied on class probabilities
    so that the comparison remains consistent across ablations.
    """
    def __init__(self, backbone, alpha_ce=ALPHA_CE, beta_margin=BETA_MARGIN, **kwargs):
        super().__init__(**kwargs)
        self.backbone = backbone
        self.alpha_ce = alpha_ce
        self.beta_margin = beta_margin
        self.ce_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)
        self.loss_tracker = tf.keras.metrics.Mean(name='loss')
        self.ce_tracker = tf.keras.metrics.Mean(name='ce_loss')
        self.margin_tracker = tf.keras.metrics.Mean(name='margin_loss')
        self.acc_metric = tf.keras.metrics.CategoricalAccuracy(name='accuracy')
        self.auc_metric = AUC(name='auc', curve='ROC', multi_label=False)

    @property
    def metrics(self):
        return [self.loss_tracker, self.ce_tracker, self.margin_tracker, self.acc_metric, self.auc_metric]

    def capsule_margin_loss(self, y_true, y_pred):
        present_error = tf.square(tf.maximum(0.0, 0.9 - y_pred))
        absent_error = tf.square(tf.maximum(0.0, y_pred - 0.1))
        loss = y_true * present_error + 0.5 * (1.0 - y_true) * absent_error
        return tf.reduce_mean(tf.reduce_sum(loss, axis=1))

    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            y_pred = self.backbone(x, training=True)
            ce_loss = self.ce_fn(y, y_pred)
            margin_loss = self.capsule_margin_loss(y, y_pred)
            total_loss = self.alpha_ce * ce_loss + self.beta_margin * margin_loss
            if self.backbone.losses:
                total_loss += tf.add_n(self.backbone.losses)
        grads = tape.gradient(total_loss, self.backbone.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.backbone.trainable_variables))
        self.loss_tracker.update_state(total_loss)
        self.ce_tracker.update_state(ce_loss)
        self.margin_tracker.update_state(margin_loss)
        self.acc_metric.update_state(y, y_pred)
        self.auc_metric.update_state(y, y_pred)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        x, y = data
        y_pred = self.backbone(x, training=False)
        ce_loss = self.ce_fn(y, y_pred)
        margin_loss = self.capsule_margin_loss(y, y_pred)
        total_loss = self.alpha_ce * ce_loss + self.beta_margin * margin_loss
        if self.backbone.losses:
            total_loss += tf.add_n(self.backbone.losses)
        self.loss_tracker.update_state(total_loss)
        self.ce_tracker.update_state(ce_loss)
        self.margin_tracker.update_state(margin_loss)
        self.acc_metric.update_state(y, y_pred)
        self.auc_metric.update_state(y, y_pred)
        return {m.name: m.result() for m in self.metrics}

    def call(self, x, training=False, return_extras=False):
        return self.backbone(x, training=training, return_extras=return_extras)


def build_backbone(architecture, num_classes):
    if architecture == 'mobilenetv2_only':
        return MobileNetV2Only(num_classes=num_classes, name='MobileNetV2_Only')
    if architecture == 'mobilenetv2_transformer':
        return MobileNetV2Transformer(num_classes=num_classes, name='MobileNetV2_Transformer_NoCapsule')
    if architecture == 'mobilenetv2_capsule':
        return MobileNetV2Capsule(num_classes=num_classes, name='MobileNetV2_Capsule_NoTransformer')
    if architecture == 'mobilenetv2_transformer_capsule':
        return MobileNetV2TransformerCapsule(num_classes=num_classes, name='MobileNetV2_Transformer_Capsule')
    raise ValueError(f"Unknown architecture: {architecture}")


def compile_model(model, use_custom_loss, lr):
    optimizer = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=WEIGHT_DECAY)
    if use_custom_loss:
        model.compile(optimizer=optimizer)
    else:
        model.compile(
            optimizer=optimizer,
            loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
            metrics=[tf.keras.metrics.CategoricalAccuracy(name='accuracy'), AUC(name='auc', curve='ROC', multi_label=False)]
        )


def get_backbone_from_model(model, use_custom_loss):
    return model.backbone if use_custom_loss else model


def train_two_phase(architecture, loss_mode, train_gen, val_gen, num_classes):
    use_custom_loss = loss_mode == 'custom'
    backbone = build_backbone(architecture, num_classes)
    model = CustomLossTrainer(backbone, name=f'{architecture}_CustomLoss_Trainer') if use_custom_loss else backbone

    # Build weights before summary/training
    _ = model(tf.random.normal((2, 224, 224, 3)), training=False)
    print("\n" + "=" * 80)
    print(f"Training architecture: {architecture} | loss_mode: {loss_mode}")
    print("=" * 80)
    model.summary()

    # Phase 1: train only top layers; MobileNetV2 stays frozen
    compile_model(model, use_custom_loss=use_custom_loss, lr=PHASE1_LR)
    ckpt1 = f'/content/working/{architecture}_{loss_mode}_phase1.weights.h5'
    callbacks1 = [
        EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, mode='min'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1, mode='min'),
        ModelCheckpoint(ckpt1, monitor='val_loss', mode='min', save_best_only=True, save_weights_only=True, verbose=1)
    ]
    history1 = model.fit(train_gen, validation_data=val_gen, epochs=PHASE1_EPOCHS, callbacks=callbacks1, verbose=1)

    # Phase 2: fine-tune upper MobileNetV2 layers
    arch_model = get_backbone_from_model(model, use_custom_loss)
    arch_model.backbone.trainable = True
    for layer in arch_model.backbone.layers[:100]:
        layer.trainable = False

    compile_model(model, use_custom_loss=use_custom_loss, lr=PHASE2_LR)
    ckpt2 = f'/content/working/{architecture}_{loss_mode}_phase2.weights.h5'
    callbacks2 = [
        EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, mode='min'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=4, min_lr=1e-7, verbose=1, mode='min'),
        ModelCheckpoint(ckpt2, monitor='val_loss', mode='min', save_best_only=True, save_weights_only=True, verbose=1)
    ]
    history2 = model.fit(train_gen, validation_data=val_gen, epochs=PHASE2_EPOCHS, callbacks=callbacks2, verbose=1)

    if os.path.exists(ckpt2):
        model.load_weights(ckpt2)
    elif os.path.exists(ckpt1):
        model.load_weights(ckpt1)

    return model, history1, history2


def plot_training_history(history1, history2, title):
    def get_metric(name):
        return history1.history.get(name, []) + history2.history.get(name, [])

    acc = get_metric('accuracy')
    val_acc = get_metric('val_accuracy')
    loss = get_metric('loss')
    val_loss = get_metric('val_loss')
    auc = get_metric('auc')
    val_auc = get_metric('val_auc')
    split_epoch = len(history1.history.get('accuracy', []))

    plt.figure(figsize=(18, 5))
    plt.subplot(1, 3, 1)
    plt.plot(acc, label='train_acc')
    plt.plot(val_acc, label='val_acc')
    plt.axvline(split_epoch - 1, color='gray', linestyle='--')
    plt.title(f'{title} Accuracy')
    plt.legend()

    plt.subplot(1, 3, 2)
    plt.plot(loss, label='train_loss')
    plt.plot(val_loss, label='val_loss')
    plt.axvline(split_epoch - 1, color='gray', linestyle='--')
    plt.title(f'{title} Loss')
    plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(auc, label='train_auc')
    plt.plot(val_auc, label='val_auc')
    plt.axvline(split_epoch - 1, color='gray', linestyle='--')
    plt.title(f'{title} AUC')
    plt.legend()
    plt.tight_layout()
    plt.show()


def evaluate_model(model, test_gen, class_names, title):
    test_gen.reset()
    y_true = test_gen.classes
    y_prob = model.predict(test_gen, verbose=1)
    y_prob = y_prob[:len(y_true)]
    y_pred = np.argmax(y_prob, axis=1)

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    y_true_onehot = to_categorical(y_true, num_classes=len(class_names))
    try:
        auc = roc_auc_score(y_true_onehot, y_prob, multi_class='ovr')
    except Exception as e:
        auc = np.nan
        print("AUC calculation skipped:", e)

    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

    print(f"Accuracy: {acc:.4f}")
    print(f"Precision-weighted: {precision:.4f}")
    print(f"Recall-weighted: {recall:.4f}")
    print(f"F1-weighted: {f1:.4f}")
    print(f"AUC-ROC OvR: {auc:.4f}" if not np.isnan(auc) else "AUC-ROC OvR: NaN")

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'{title} Confusion Matrix')
    plt.tight_layout()
    plt.show()

    return {
        'title': title,
        'accuracy': acc,
        'precision_weighted': precision,
        'recall_weighted': recall,
        'f1_weighted': f1,
        'auc_ovr': auc
    }, y_true, y_pred, y_prob


In [ ]:
# ============================================================
# 8. Run experiment(s)
# ============================================================
# Architectures available:
# mobilenetv2_only = MobileNetV2 CNN only
# mobilenetv2_transformer = MobileNetV2 + custom Transformer, no Capsule
# mobilenetv2_capsule = MobileNetV2 + Capsule, no Transformer
# mobilenetv2_transformer_capsule = MobileNetV2 + Transformer + Capsule

ARCHITECTURES_TO_RUN = ['mobilenetv2_only', 'mobilenetv2_transformer', 'mobilenetv2_capsule', 'mobilenetv2_transformer_capsule']
LOSS_MODES_TO_RUN = ['ce', 'custom']  # ce = standard CE; custom = CE + capsule/margin loss

all_results = []
trained_models = {}
histories = {}

for architecture in ARCHITECTURES_TO_RUN:
    for loss_mode in LOSS_MODES_TO_RUN:
        tf.keras.backend.clear_session()
        model, history1, history2 = train_two_phase(architecture, loss_mode, train_gen, val_gen, NUM_CLASSES)
        title = f"{architecture} | {loss_mode}"
        plot_training_history(history1, history2, title=title)
        result, y_true, y_pred, y_prob = evaluate_model(model, test_gen, CLASS_NAMES, title=title)
        result['architecture'] = architecture
        result['loss_mode'] = loss_mode
        all_results.append(result)
        trained_models[(architecture, loss_mode)] = model
        histories[(architecture, loss_mode)] = (history1, history2)

results_df = pd.DataFrame(all_results)
results_df = results_df[['architecture', 'loss_mode', 'accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted', 'auc_ovr', 'title']]
results_path = '/content/working/ablation_results.csv'
results_df.to_csv(results_path, index=False)
display(results_df)
print('Saved results to:', results_path)


In [ ]:
# ============================================================
# 9. Architecture note for thesis/paper writing
# ============================================================
architecture_notes = pd.DataFrame([
    {
        'file/model': 'MobileNetV2 only',
        'CNN': 'Yes, MobileNetV2 ImageNet backbone',
        'Transformer': 'No',
        'Capsule': 'No',
        'Loss comparison': 'Standard CE vs CE + margin-style custom loss'
    },
    {
        'file/model': 'MobileNetV2 + Transformer',
        'CNN': 'Yes, MobileNetV2 ImageNet backbone',
        'Transformer': 'Yes, custom Transformer encoder with MultiHeadAttention',
        'Capsule': 'No',
        'Loss comparison': 'Standard CE vs CE + margin-style custom loss'
    },
    {
        'file/model': 'MobileNetV2 + Capsule',
        'CNN': 'Yes, MobileNetV2 ImageNet backbone',
        'Transformer': 'No',
        'Capsule': 'Yes, PrimaryCapsule + RelevanceAwareClassCapsule',
        'Loss comparison': 'Standard CE vs CE + Capsule/Margin loss'
    },
    {
        'file/model': 'MobileNetV2 + Transformer + Capsule',
        'CNN': 'Yes, MobileNetV2 ImageNet backbone',
        'Transformer': 'Yes, custom Transformer encoder',
        'Capsule': 'Yes, relevance-aware capsule classifier',
        'Loss comparison': 'Standard CE vs CE + Capsule/Margin loss'
    }
])
display(architecture_notes)
